In [6]:
import subprocess
import sys

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    print("✅ vaderSentiment already available")
except ImportError:
    print("📦 Installing vaderSentiment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "vaderSentiment"])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    print("✅ vaderSentiment installed and imported")

StatementMeta(, adad2db5-aeae-4eea-bb63-03f512f9d86e, 8, Finished, Available, Finished, False)

✅ vaderSentiment already available


In [7]:
from pyspark.sql.functions import current_timestamp, when, col, lit
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Enable schema evolution
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ----------------------------------------------------------------------
# Read Silver into the driver as a regular Python list
# (small data — only ~18-50 rows, no need to distribute)
# ----------------------------------------------------------------------
df_silver = spark.read.table("top_headlines_silver")
silver_rows = df_silver.collect()
print(f"Read {len(silver_rows)} rows from Silver")

# ----------------------------------------------------------------------
# Score each row on the driver (where vaderSentiment is installed)
# ----------------------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()
scored_data = []

for row in silver_rows:
    title = row["title"] or ""
    scores = analyzer.polarity_scores(title)
    
    compound = float(scores["compound"])
    if compound > 0.05:
        label = "Positive"
    elif compound < -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    
    scored_data.append({
        "title":              row["title"],
        "source_name":        row["source_name"],
        "domain":             row["domain"],
        "category":           row["category"],
        "author":             row["author"],
        "description":        row["description"],
        "url":                row["url"],
        "published_ts":       row["published_ts"],
        "ingested_at":        row["ingested_at"],
        "sentiment_compound": compound,
        "sentiment_positive": float(scores["pos"]),
        "sentiment_neutral":  float(scores["neu"]),
        "sentiment_negative": float(scores["neg"]),
        "sentiment_label":    label,
    })

print(f"Scored {len(scored_data)} rows")

# ----------------------------------------------------------------------
# Convert back to Spark DataFrame and save
# ----------------------------------------------------------------------
df_scored = spark.createDataFrame(scored_data).withColumn("scored_at", current_timestamp())

print("\nSample scored output:")
df_scored.select("title", "sentiment_label", "sentiment_compound").show(5, truncate=70)

print("\nSentiment distribution:")
df_scored.groupBy("sentiment_label").count().orderBy("count", ascending=False).show()

# ----------------------------------------------------------------------
# Save as Delta table (overwrite — derived from Silver)
# ----------------------------------------------------------------------
table_name = "top_headlines_sentiment"

(df_scored.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name))

print(f"\n✅ Saved {df_scored.count()} rows to {table_name}")

StatementMeta(, adad2db5-aeae-4eea-bb63-03f512f9d86e, 9, Finished, Available, Finished, False)

Read 17 rows from Silver
Scored 17 rows

Sample scored output:
+----------------------------------------------------------------------+---------------+------------------+
|                                                                 title|sentiment_label|sentiment_compound|
+----------------------------------------------------------------------+---------------+------------------+
|Ayo Adebiri, Bradley Cooper, Werner Herzog Join Bong Joon Ho's Ally...|       Positive|             0.296|
|'American Idol' finale 2026: Did Hannah Harper, Jordan McCullough o...|       Positive|            0.5859|
|Trump and Xi appear intent on keeping deep differences over Iran wa...|       Negative|           -0.5994|
|Lakers eliminated by Thunder after crushing Game 4 loss as LeBron J...|       Negative|           -0.7845|
|Hayden Panettiere Was Told to Get in Bed With a Famous Male Actor W...|       Negative|           -0.1027|
+----------------------------------------------------------------------+-